In [35]:
import pandas as pd
import numpy as np 

filtered_dataset = pd.read_csv(
    "data/filtered_dataset.csv"
)

In [36]:
filtered_dataset

,country,year,iso_code,population,gdp,cement_co2,cement_co2_per_capita,co2,co2_growth_abs,co2_growth_prct,...,share_global_other_co2,share_of_temperature_change_from_ghg,temperature_change_from_ch4,temperature_change_from_co2,temperature_change_from_ghg,temperature_change_from_n2o,total_ghg,total_ghg_excluding_lucf,trade_co2,trade_co2_share
0,Afghanistan,1991,AFG,12238879.0,1.204736e+10,0.046,0.004,1.914,-0.110,-5.435,...,NaN,0.088,0.000,0.000,0.001,0.0,13.711,3.038,NaN,NaN
1,Afghanistan,1992,AFG,13278983.0,1.267754e+10,0.046,0.003,1.482,-0.432,-22.580,...,NaN,0.087,0.000,0.000,0.001,0.0,11.543,2.606,NaN,NaN
2,Afghanistan,1993,AFG,14943175.0,9.834582e+09,0.047,0.003,1.487,0.005,0.330,...,NaN,0.085,0.000,0.000,0.001,0.0,10.454,2.664,NaN,NaN
3,Afghanistan,1994,AFG,16250800.0,7.919856e+09,0.047,0.003,1.454,-0.033,-2.227,...,NaN,0.084,0.000,0.000,0.001,0.0,10.980,2.697,NaN,NaN
4,Afghanistan,1995,AFG,17065837.0,1.230753e+10,0.047,0.003,1.417,-0.037,-2.511,...,NaN,0.082,0.000,0.000,0.001,0.0,12.110,2.730,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7407,Zimbabwe,2020,ZWE,15526887.0,2.317871e+10,0.496,0.032,8.491,-1.776,-17.298,...,NaN,0.106,0.001,0.001,0.002,0.0,24.146,14.463,1.991,23.450
7408,Zimbabwe,2021,ZWE,15797220.0,2.514009e+10,0.542,0.034,10.223,1.732,20.398,...,NaN,0.105,0.001,0.001,0.002,0.0,27.907,16.408,2.137,20.899
7409,Zimbabwe,2022,ZWE,16069061.0,2.590159e+10,0.387,0.024,12.232,2.009,19.653,...,NaN,0.104,0.001,0.001,0.002,0.0,29.917,18.830,1.380,11.283
7410,Zimbabwe,2023,ZWE,16340829.0,NaN,0.387,0.024,13.443,1.211,9.904,...,NaN,0.103,0.001,0.001,0.002,0.0,31.029,20.318,1.876,13.957


In [37]:
filtered_dataset["decade"] = (
    filtered_dataset["year"] // 10
) * 10

vectorization is done on the entire column year and new column is formed that is decade

In [38]:
filtered_dataset[["year", "decade"]].head(5)

,year,decade
0,1991,1990
1,1992,1990
2,1993,1990
3,1994,1990
4,1995,1990


In [39]:
filtered_dataset["years_since_1990"] = (
    filtered_dataset["year"] - 1990
)

A new feature called `years_since_1990` is created by subtracting 1990 from the `year` column. This converts calendar years into a simple numerical time index, making it easier for regression models to learn temporal relationships while preserving the order of observations.

In [40]:
filtered_dataset[
    ["year","years_since_1990"]
].head(5)

,year,years_since_1990
0,1991,1
1,1992,2
2,1993,3
3,1994,4
4,1995,5


In [41]:
filtered_dataset["co2_5yr_rolling_mean"] = (
    filtered_dataset
    .groupby("country")["co2"]
    .transform(
        lambda x: x.rolling(window=5).mean()
    )
)

##### Creating the 5-Year Rolling Mean Feature

A new feature, co2_5yr_rolling_mean, captures the long-term trend of CO₂ emissions by calculating the average emissions over the current year and the previous four years. The dataset is grouped by country so the rolling mean is computed independently for each country. Using rolling(window=5).mean() smooths short-term fluctuations and highlights long-term trends, making the data more suitable for time-series analysis and machine learning.

In [42]:
filtered_dataset[
    [
        "country",
        "year",
        "co2",
        "co2_5yr_rolling_mean"
    ]
].head()

,country,year,co2,co2_5yr_rolling_mean
0,Afghanistan,1991,1.914,NaN
1,Afghanistan,1992,1.482,NaN
2,Afghanistan,1993,1.487,NaN
3,Afghanistan,1994,1.454,NaN
4,Afghanistan,1995,1.417,1.5508


In [43]:
filtered_dataset["co2_lag1"] = (
    filtered_dataset
    .groupby("country")["co2"]
    .shift(1)
)

filtered_dataset["co2_lag2"] = (
    filtered_dataset
    .groupby("country")["co2"]
    .shift(2)
)

filtered_dataset["co2_lag3"] = (
    filtered_dataset
    .groupby("country")["co2"]
    .shift(3)
)

##### Creating Lag Features

Lag features capture previous years' CO₂ emissions, helping the model learn temporal patterns.

Three lag features are created:

1) co2_lag1: Previous year's CO₂ emission
2) co2_lag2: CO₂ emission from two years ago
3) co2_lag3: CO₂ emission from three years ago

The data is grouped by country, and shift() is used to create these features without mixing countries.

In [44]:
filtered_dataset[
    [
        "country",
        "year",
        "co2",
        "co2_lag1",
        "co2_lag2",
        "co2_lag3"
    ]
].head(15)

,country,year,co2,co2_lag1,co2_lag2,co2_lag3
0,Afghanistan,1991,1.914,NaN,NaN,NaN
1,Afghanistan,1992,1.482,1.914,NaN,NaN
2,Afghanistan,1993,1.487,1.482,1.914,NaN
3,Afghanistan,1994,1.454,1.487,1.482,1.914
4,Afghanistan,1995,1.417,1.454,1.487,1.482
5,Afghanistan,1996,1.370,1.417,1.454,1.487
6,Afghanistan,1997,1.304,1.370,1.417,1.454
7,Afghanistan,1998,1.279,1.304,1.370,1.417
8,Afghanistan,1999,1.092,1.279,1.304,1.370
9,Afghanistan,2000,1.047,1.092,1.279,1.304


In [45]:
filtered_dataset.columns.tolist()

['country',
 'year',
 'iso_code',
 'population',
 'gdp',
 'cement_co2',
 'cement_co2_per_capita',
 'co2',
 'co2_growth_abs',
 'co2_growth_prct',
 'co2_including_luc',
 'co2_including_luc_growth_abs',
 'co2_including_luc_growth_prct',
 'co2_including_luc_per_capita',
 'co2_including_luc_per_gdp',
 'co2_including_luc_per_unit_energy',
 'co2_per_capita',
 'co2_per_gdp',
 'co2_per_unit_energy',
 'coal_co2',
 'coal_co2_per_capita',
 'consumption_co2',
 'consumption_co2_per_capita',
 'consumption_co2_per_gdp',
 'cumulative_cement_co2',
 'cumulative_co2',
 'cumulative_co2_including_luc',
 'cumulative_coal_co2',
 'cumulative_flaring_co2',
 'cumulative_gas_co2',
 'cumulative_luc_co2',
 'cumulative_oil_co2',
 'cumulative_other_co2',
 'energy_per_capita',
 'energy_per_gdp',
 'flaring_co2',
 'flaring_co2_per_capita',
 'gas_co2',
 'gas_co2_per_capita',
 'ghg_excluding_lucf_per_capita',
 'ghg_per_capita',
 'land_use_change_co2',
 'land_use_change_co2_per_capita',
 'methane',
 'methane_per_capita',
 

In [46]:
filtered_dataset["calculated_co2_per_capita"] = (
    (filtered_dataset["co2"]*1000000) /
    filtered_dataset["population"]
)

##### Verifying the CO₂ Per Capita Feature

The existing `co2_per_capita` feature is verified by recalculating it using the `co2` and `population` columns. The values are compared after rounding to account for minor floating-point differences, and a sample of three countries across three years is displayed for validation.

In [61]:


comparison = pd.Series(np.isclose(
    filtered_dataset["co2_per_capita"],
    filtered_dataset["calculated_co2_per_capita"],
    atol=0.001)
)

comparison.value_counts()

True     6089
False    1323
Name: count, dtype: int64

In [72]:
mismatch = filtered_dataset[
    comparison == False
]

mismatch[
    [
        "country",
        "year",
        "co2",
        "population",
        "co2_per_capita",
        "calculated_co2_per_capita"
    ]
].head(20)

,country,year,co2,population,co2_per_capita,calculated_co2_per_capita
104,Andorra,1993,0.407,63293.0,6.426,6.430411
107,Andorra,1996,0.454,64006.0,7.098,7.093085
108,Andorra,1997,0.465,64722.0,7.190,7.184574
113,Andorra,2002,0.531,66528.0,7.986,7.981602
115,Andorra,2004,0.564,74342.0,7.590,7.586559
116,Andorra,2005,0.579,77436.0,7.476,7.477142
118,Andorra,2007,0.539,81901.0,6.576,6.581116
119,Andorra,2008,0.539,83513.0,6.449,6.454085
120,Andorra,2009,0.517,83907.0,6.157,6.161584
121,Andorra,2010,0.517,80727.0,6.400,6.404301


The recalculated values closely match the provided feature. Minor discrepancies are attributed to rounding and precision differences in the source dataset

In [70]:
verification_sample = filtered_dataset[
    (
        filtered_dataset["country"].isin(
            ["India", "China", "United States"]
        )
    )
    &
    (
        filtered_dataset["year"].isin(
            [1995, 2005, 2015]
        )
    )
]

verification_sample[
    [
        "country",
        "year",
        "co2",
        "population",
        "co2_per_capita",
        "calculated_co2_per_capita"
    ]
]

,country,year,co2,population,co2_per_capita,calculated_co2_per_capita
1398,China,1995,3351.197,1.220134e+09,2.747,2.746581
1408,China,2005,5881.991,1.310027e+09,4.490,4.489976
1418,China,2015,9858.040,1.396134e+09,7.061,7.060955
3064,India,1995,760.480,9.603010e+08,0.792,0.791918
3074,India,2005,1195.393,1.154676e+09,1.035,1.035262
3084,India,2015,2231.817,1.328024e+09,1.681,1.680554
7042,United States,1995,5425.837,2.682058e+08,20.230,20.230126
7052,United States,2005,6126.903,2.957167e+08,20.719,20.718829
7062,United States,2015,5368.497,3.261265e+08,16.461,16.461395


##### Observation

The `co2_per_capita` feature was verified by recalculating CO₂ emissions per person using the `co2` and `population` columns for three countries across three different years. The recalculated values closely match the existing dataset values, with only very small differences caused by floating-point precision and rounding of the displayed data. This confirms that the `co2_per_capita` feature is correctly computed and can be used for further analysis.

In [73]:
filtered_dataset["ghg_intensity"] = (
    filtered_dataset["total_ghg"] /
    filtered_dataset["gdp"]
)

In [74]:
filtered_dataset[
    [
        "country",
        "year",
        "total_ghg",
        "gdp",
        "ghg_intensity"
    ]
].head(10)

,country,year,total_ghg,gdp,ghg_intensity
0,Afghanistan,1991,13.711,1.204736e+10,1.138091e-09
1,Afghanistan,1992,11.543,1.267754e+10,9.105079e-10
2,Afghanistan,1993,10.454,9.834582e+09,1.062984e-09
3,Afghanistan,1994,10.980,7.919856e+09,1.386389e-09
4,Afghanistan,1995,12.110,1.230753e+10,9.839508e-10
5,Afghanistan,1996,14.851,1.207013e+10,1.230393e-09
6,Afghanistan,1997,18.057,1.185075e+10,1.523701e-09
7,Afghanistan,1998,16.691,1.169217e+10,1.427536e-09
8,Afghanistan,1999,18.338,1.151732e+10,1.592211e-09
9,Afghanistan,2000,16.009,1.128379e+10,1.418760e-09


##### Creating the GHG Intensity Feature

A new feature called `ghg_intensity` is created to measure the amount of greenhouse gas emissions generated for each unit of economic output.

The feature is calculated by dividing `total_ghg` (measured in million tonnes) by `gdp` (measured in US dollars). Lower GHG intensity indicates a more carbon-efficient economy because it produces greater economic output for the same level of greenhouse gas emissions.

The feature is calculated only where both `total_ghg` and `gdp` values are available. Rows with missing values remain as `NaN`.

In [79]:
missing_gdp = filtered_dataset[
    filtered_dataset["gdp"].isna()
]

missing_gdp[
    [
        "country",
        "year"
    ]
].sort_values(["country","year"])

,country,year
32,Afghanistan,2023
33,Afghanistan,2024
66,Albania,2023
67,Albania,2024
100,Algeria,2023
...,...,...
7343,Yemen,2024
7376,Zambia,2023
7377,Zambia,2024
7410,Zimbabwe,2023


##### Identifying Missing GDP Values

GHG intensity cannot be calculated when GDP data is unavailable. To identify these cases, rows with missing GDP values are extracted and displayed. This helps assess data completeness and explains why some `ghg_intensity` values remain missing.

In [80]:
filtered_dataset["co2_yoy_change"] = (
    filtered_dataset
    .groupby("country")["co2"]
    .diff()
)

##### Creating the Year-over-Year CO₂ Change Feature

A new feature called `co2_yoy_change` is created to measure the absolute change in CO₂ emissions from one year to the next for each country.

The feature is calculated by subtracting the previous year's CO₂ emissions from the current year's emissions using the `diff()` function. The dataset is grouped by country to ensure that changes are calculated independently for each country and not across different countries.

The first year for each country contains a `NaN` value because there is no previous year's data available for comparison.

In [81]:
filtered_dataset[
    [
        "country",
        "year",
        "co2",
        "co2_yoy_change"
    ]
].head(15)

,country,year,co2,co2_yoy_change
0,Afghanistan,1991,1.914,NaN
1,Afghanistan,1992,1.482,-0.432
2,Afghanistan,1993,1.487,0.005
3,Afghanistan,1994,1.454,-0.033
4,Afghanistan,1995,1.417,-0.037
5,Afghanistan,1996,1.370,-0.047
6,Afghanistan,1997,1.304,-0.066
7,Afghanistan,1998,1.279,-0.025
8,Afghanistan,1999,1.092,-0.187
9,Afghanistan,2000,1.047,-0.045


In [83]:
filtered_dataset["co2_yoy_pct_change"] = (
    filtered_dataset
    .groupby("country")["co2"]
    .pct_change() * 100
)

C:\Users\club laptop\AppData\Local\Temp\ipykernel_2644\1523492477.py:4: FutureWarning: The default fill_method='ffill' in SeriesGroupBy.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  .pct_change() * 100


##### Creating the Year-over-Year Percentage Change Feature

A new feature called `co2_yoy_pct_change` is created to measure the percentage change in CO₂ emissions compared to the previous year for each country.

The feature is calculated using the `pct_change()` function, which computes the relative change between consecutive years. The dataset is grouped by country to ensure that percentage changes are calculated independently for each country.

The first observation for each country contains a `NaN` value because there is no previous year's data available for comparison.

In [84]:
filtered_dataset[
    [
        "country",
        "year",
        "co2",
        "co2_yoy_change",
        "co2_yoy_pct_change"
    ]
].head(15)

,country,year,co2,co2_yoy_change,co2_yoy_pct_change
0,Afghanistan,1991,1.914,NaN,NaN
1,Afghanistan,1992,1.482,-0.432,-22.570533
2,Afghanistan,1993,1.487,0.005,0.337382
3,Afghanistan,1994,1.454,-0.033,-2.219233
4,Afghanistan,1995,1.417,-0.037,-2.544704
5,Afghanistan,1996,1.370,-0.047,-3.316867
6,Afghanistan,1997,1.304,-0.066,-4.817518
7,Afghanistan,1998,1.279,-0.025,-1.917178
8,Afghanistan,1999,1.092,-0.187,-14.620797
9,Afghanistan,2000,1.047,-0.045,-4.120879


In [92]:


filtered_dataset["co2_yoy_pct_change"] = (
    filtered_dataset["co2_yoy_pct_change"]
    .replace([np.inf, -np.inf], np.nan)
)

pct_change() divides the current value difference by the previous value. If the previous value is zero, the calculation involves division by zero, which results in positive or negative infinity (inf or -inf). These values are typically replaced with NaN before further analysis.

In [93]:
top_growth_countries = (
    filtered_dataset
    .groupby("country")["co2_yoy_pct_change"]
    .mean()
    .sort_values(
        ascending=False
    )
    .head(5)
)

top_growth_countries

country
Equatorial Guinea                  54.087422
Sint Maarten (Dutch part)          23.740771
Bonaire Sint Eustatius and Saba    23.167343
Curacao                            21.409001
East Timor                         16.026552
Name: co2_yoy_pct_change, dtype: float64

###### Observation

The table shows the five countries with the highest average annual percentage growth in CO₂ emissions since 1990. Most of these are small countries or territories where even small increases in emissions result in large percentage changes. Therefore, percentage growth should be interpreted alongside absolute emission values for a complete understanding of emission trends.

In [94]:
top_reduction_countries = (
    filtered_dataset
    .groupby("country")["co2_yoy_change"]
    .mean()
    .sort_values(
        ascending=True
    )
    .head(5)
)

top_reduction_countries

country
Russia           -19.003576
Ukraine          -14.869576
Germany          -13.473061
Kuwait           -11.013939
United Kingdom    -8.985061
Name: co2_yoy_change, dtype: float64

##### Identifying the Top 5 Countries with the Largest Average Annual CO₂ Reductions

The annual absolute change in CO₂ emissions is averaged for each country to identify long-term reduction trends. Countries with the most negative average year-over-year CO₂ change represent those that have consistently reduced emissions since 1990.

The countries are sorted in ascending order because larger reductions correspond to more negative values.

In [95]:
feature_columns = [
    "country",
    "year",
    "co2",
    "co2_per_capita",
    "co2_5yr_rolling_mean",
    "co2_lag1",
    "co2_lag2",
    "co2_lag3",
    "co2_yoy_pct_change",
    "ghg_intensity"
]

ghg_features = filtered_dataset[feature_columns]

In [96]:
ghg_features.head()

,country,year,co2,co2_per_capita,co2_5yr_rolling_mean,co2_lag1,co2_lag2,co2_lag3,co2_yoy_pct_change,ghg_intensity
0,Afghanistan,1991,1.914,0.156,NaN,NaN,NaN,NaN,NaN,1.138091e-09
1,Afghanistan,1992,1.482,0.112,NaN,1.914,NaN,NaN,-22.570533,9.105079e-10
2,Afghanistan,1993,1.487,0.100,NaN,1.482,1.914,NaN,0.337382,1.062984e-09
3,Afghanistan,1994,1.454,0.089,NaN,1.487,1.482,1.914,-2.219233,1.386389e-09
4,Afghanistan,1995,1.417,0.083,1.5508,1.454,1.487,1.482,-2.544704,9.839508e-10


In [97]:
ghg_features.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7412 entries, 0 to 7411
Data columns (total 10 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   country               7412 non-null   object 
 1   year                  7412 non-null   int64  
 2   co2                   7299 non-null   float64
 3   co2_per_capita        7234 non-null   float64
 4   co2_5yr_rolling_mean  6439 non-null   float64
 5   co2_lag1              7084 non-null   float64
 6   co2_lag2              6869 non-null   float64
 7   co2_lag3              6654 non-null   float64
 8   co2_yoy_pct_change    7034 non-null   float64
 9   ghg_intensity         5214 non-null   float64
dtypes: float64(8), int64(1), object(1)
memory usage: 579.2+ KB


In [98]:
ghg_features.isna().sum()

country                    0
year                       0
co2                      113
co2_per_capita           178
co2_5yr_rolling_mean     973
co2_lag1                 328
co2_lag2                 543
co2_lag3                 758
co2_yoy_pct_change       378
ghg_intensity           2198
dtype: int64

In [99]:
ghg_features.to_csv(
    "data/ghg_features.csv",
    index=False
)

##### Creating the Final Feature Dataset

After completing the feature engineering process, a clean modelling dataset is created by selecting only the columns required for machine learning.

The dataset includes the target variable (`co2`) and the engineered features developed throughout Week 2. This final dataset will be used for training and evaluating machine learning models in the next stage of the project.

The `ghg_intensity` feature is included wherever GDP data is available. Missing values resulting from lag features, rolling averages, or unavailable GDP data are retained because they are expected outcomes of the feature engineering process.